# Local experiment
Runs on CPU/Mac. `TINY = True` uses a tiny random model + fake data (offline, seconds) to check the pipeline.
Set `TINY = False` for the real Qwen + GSM8K (downloads ~1GB; slow on CPU — real runs go on Kaggle).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")  # repo root

import torch
from torch.optim import AdamW
from model import get_device, load_tokenizer, load_model, tiny_model, print_trainable_parameters
from data import load_gsm8k, tiny_dataset, make_loader
from train import train_one_epoch, eval_val_loss, forward_loss
from evaluation import evaluate_gsm8k, show_samples

## cfg

In [ ]:
TINY = True           # tiny offline model + fake data
DEBUG = True          # 3 steps, shape printing, grad check

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
BATCH_SIZE = 2
LR = 1e-3 if TINY else 3e-5
NUM_EPOCHS = 1
MAX_LENGTH, MAX_INPUT_LENGTH = 384, 128

torch.manual_seed(0)
device = get_device()
print("Device:", device)

## model + data

In [ ]:
if TINY:
    tokenizer = load_tokenizer("openai-community/gpt2")
    llm = tiny_model(len(tokenizer)).to(device)
    print_trainable_parameters(llm)
    train_ds = val_ds = test_ds = tiny_dataset(8)
else:
    tokenizer = load_tokenizer(MODEL_NAME)
    llm = load_model(MODEL_NAME, device, lora=True)
    train_ds, val_ds, test_ds = load_gsm8k(val_size=256)

train_loader = make_loader(train_ds, tokenizer, BATCH_SIZE, MAX_LENGTH, MAX_INPUT_LENGTH)
val_loader = make_loader(val_ds, tokenizer, BATCH_SIZE, MAX_LENGTH, MAX_INPUT_LENGTH, shuffle=False)
optimizer = AdamW([p for p in llm.parameters() if p.requires_grad], lr=LR)

## inspect one batch

In [ ]:
batch = next(iter(train_loader))
loss, logits = forward_loss(llm, batch, device, debug=True)
print(tokenizer.decode(batch["input_ids"][0]))

## train

In [ ]:
for epoch in range(NUM_EPOCHS):
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    losses = train_one_epoch(llm, train_loader, optimizer, device, accum_steps=1, debug=DEBUG)
    print("val loss:", eval_val_loss(llm, val_loader, device, debug=DEBUG))

## eval

In [ ]:
show_samples(llm, tokenizer, test_ds, device, n=2, max_new_tokens=16)
acc, preds = evaluate_gsm8k(llm, tokenizer, test_ds, device, batch_size=BATCH_SIZE, debug=DEBUG)

## scratch